# 04 - SHAP Explainability
**Project**: Interpretable Student Academic Risk Prediction  
**Course**: CSE2267 – Machine Learning Techniques

This notebook provides global and local explanations of the best model using SHAP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

import shap
from sklearn.model_selection import train_test_split

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries imported successfully!")
shap.initjs()

## 1. Load Data and Best Model

In [ ]:
# Load data (same logic as modeling notebook)
processed_path = "../data/processed/student_processed.csv"
raw_path = "../data/student-mat.csv"

if os.path.exists(processed_path):
    df = pd.read_csv(processed_path)
    print("Loaded preprocessed data")
else:
    df = pd.read_csv(raw_path)
    df['At_Risk'] = (df['G3'] < 10).astype(int)
    cols_to_drop = ['G1', 'G2', 'G3']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    print("Loaded raw data + basic preprocessing")

TARGET = 'At_Risk'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Data loaded. Shape:", X.shape)

In [ ]:
# Load best model
model_dir = "../results/models/"
model_files = [f for f in os.listdir(model_dir) if f.startswith("best_model") and f.endswith(".joblib")]

if not model_files:
    raise FileNotFoundError("No best model found! Please run 03_Modeling.ipynb first.")

model_path = os.path.join(model_dir, model_files[0])
model = joblib.load(model_path)
print("Loaded model from:", model_path)

# Load feature names if available
feature_names_path = os.path.join(model_dir, "feature_names.joblib")
if os.path.exists(feature_names_path):
    feature_names = joblib.load(feature_names_path)
    X_test = X_test[feature_names]  # ensure order
else:
    feature_names = list(X.columns)

print("Number of features:", len(feature_names))

## 2. Create SHAP Explainer

In [ ]:
# Use TreeExplainer for tree-based models, otherwise KernelExplainer (slower)
model_name = type(model).__name__
print("Model type:", model_name)

if model_name in ["RandomForestClassifier", "XGBClassifier", "LGBMClassifier", "DecisionTreeClassifier"]:
    explainer = shap.TreeExplainer(model)
    print("Using TreeExplainer")
else:
    # For linear models we can use LinearExplainer or KernelExplainer
    explainer = shap.Explainer(model, X_train)
    print("Using general Explainer")

# Calculate SHAP values on test set (can use a sample if dataset is large)
sample_size = min(100, len(X_test))
X_sample = X_test.iloc[:sample_size]
shap_values = explainer(X_sample)

print("SHAP values calculated for", sample_size, "samples")

## 3. Global Feature Importance (Summary Plot)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP Summary Plot (Global Feature Importance)")
plt.tight_layout()
plt.savefig("../results/figures/shap_summary_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/figures/shap_summary_plot.png")

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar Plot)")
plt.tight_layout()
plt.savefig("../results/figures/shap_bar_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/figures/shap_bar_plot.png")

## 4. Local Explanations (Individual Students)

In [ ]:
# Find one At-Risk and one Not At-Risk example
y_sample = y_test.iloc[:sample_size].reset_index(drop=True)

at_risk_idx = y_sample[y_sample == 1].index[0] if (y_sample == 1).any() else 0
not_risk_idx = y_sample[y_sample == 0].index[0] if (y_sample == 0).any() else 1

print(f"At-Risk example index   : {at_risk_idx}")
print(f"Not At-Risk example index: {not_risk_idx}")

In [ ]:
# Waterfall plot for At-Risk student
print("\n===== At-Risk Student Explanation =====")
plt.figure()
shap.plots.waterfall(shap_values[at_risk_idx], max_display=12, show=False)
plt.title("SHAP Waterfall - At-Risk Student")
plt.tight_layout()
plt.savefig("../results/figures/shap_waterfall_at_risk.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall plot for Not At-Risk student
print("\n===== Not At-Risk Student Explanation =====")
plt.figure()
shap.plots.waterfall(shap_values[not_risk_idx], max_display=12, show=False)
plt.title("SHAP Waterfall - Not At-Risk Student")
plt.tight_layout()
plt.savefig("../results/figures/shap_waterfall_not_at_risk.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Top Risk Factors (Text Summary)

In [ ]:
# Mean absolute SHAP values
if hasattr(shap_values, "values"):
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_|SHAP|": mean_abs_shap
}).sort_values("Mean_|SHAP|", ascending=False).reset_index(drop=True)

print("\nTop 15 Most Important Features according to SHAP:")
display(importance_df.head(15))

importance_df.to_csv("../results/shap_feature_importance.csv", index=False)
print("\nSaved: results/shap_feature_importance.csv")

## Summary & Insights

- Global SHAP plots show which features overall push students toward being At-Risk.
- Local waterfall plots explain individual predictions (useful for educators).
- Top risk factors can be used for early intervention strategies.

**Files generated:**
- `results/figures/shap_summary_plot.png`
- `results/figures/shap_bar_plot.png`
- `results/figures/shap_waterfall_at_risk.png`
- `results/figures/shap_waterfall_not_at_risk.png`
- `results/shap_feature_importance.csv`

This completes the core technical part of the project (Modeling + Interpretability).